In [8]:
from lancedb.rerankers import ColbertReranker
import lancedb
from dreamai.rag import add_to_lance_table, pdf_to_md_docs

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
file = "soccer.txt"

In [3]:
md_docs = pdf_to_md_docs(file)

In [15]:
md_docs.chunks[:5]

[{'text': '## Table of Contents\n1. [Introduction to Soccer](#1-introduction-to-soccer)\n2. [Basic Rules of the Game](#2-basic-rules-of-the-game)\n3. [The Playing Field](#3-the-playing-field)\n4. [Player Positions and Roles](#4-player-positions-and-roles)\n5. [Scoring and Gameplay](#5-scoring-and-gameplay)\n6. [Fouls and Misconduct](#6-fouls-and-misconduct)\n7. [Set Pieces](#7-set-pieces)\n8. [Offside Rule](#8-offside-rule)\n9. [Video Assistant Referee (VAR)](#9-video-assistant-referee-var)\n10. [Popular Formations in Modern Soccer](#10-popular-formations-in-modern-soccer)\n11. [Major Competitions and Leagues](#11-major-competitions-and-leagues)\n12. [Glossary of Soccer Terms](#12-glossary-of-soccer-terms)'},
 {'text': "## 1. Introduction to Soccer\nSoccer, also known as football in many parts of the world, is a sport played between\ntwo teams of 11 players each. The objective is to score goals by getting the ball into\nthe opposing team's goal. It is the world's most popular sport, pl

In [11]:
lance_db = lancedb.connect("lance/rag/")

In [12]:
table = add_to_lance_table(db=lance_db, table_name="soccer", data=md_docs.chunks)

/home/hamza/dev/dreamai/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
table.to_pandas()[:5]

,text,vector
0,## Table of Contents\n1. [Introduction to Socc...,"[0.016098352, 0.009591325, 0.02150682, 0.05674..."
1,"## 1. Introduction to Soccer\nSoccer, also kno...","[0.037568193, 0.019109799, 0.032952443, 0.0403..."
2,"area\n- Matches are typically 90 minutes long,...","[0.02244203, 0.014541662, 0.016999098, 0.05029..."
3,## 2. Basic Rules of the Game\nModern soccer i...,"[0.038992863, 0.018538231, 0.015996661, 0.0487..."
4,**Ball In and Out of Play**: The ball is consi...,"[0.028029257, 0.026457477, 0.02366773, 0.05125..."


In [16]:
lance_db = lancedb.connect("lance/rag/")
table = add_to_lance_table(db=lance_db, table_name="soccer", data=md_docs.chunks)
reranker = ColbertReranker("answerdotai/answerai-colbert-small-v1")
res = table.search(query=query, query_type="hybrid").rerank(reranker=reranker)

In [20]:
res.limit(3).to_pandas()

,vector,text,_relevance_score
0,"[0.015030094, 0.020254651, 0.025763832, 0.0552...",## 8. Offside Rule\nThe offside rule is one of...,0.928944
1,"[0.02059467, 0.020678537, 0.026556041, 0.07207...",**No Offside**:\n A player cannot be offside ...,0.924199
2,"[0.028029257, 0.026457477, 0.02366773, 0.05125...",**Ball In and Out of Play**: The ball is consi...,0.901171
